# C8. Baselines

Does the world beat every way of making a bounded method infinite? Needs `erosion-datagen chunked main_512 test --chunk 256` for the simulator chunks.

See `docs/plans/experiments-plan.md`, entry C8. Results: `artifacts/results/world_all_runs/`.

## Run

From the repo root. `uv run erosion-sim list` shows what is already trained and evaluated.

**1. Data.** Skip any that already exist.

Local:

```sh
(cd datagen && uv run erosion-datagen generate erosion_master/main_512)
(cd datagen && uv run erosion-datagen generate erosion_master/main_256)
(cd datagen && uv run erosion-datagen chunked erosion_master/main_512 test --chunk 256)
```

Cluster:

```sh
sbatch slurm/datagen.sbatch erosion_master/main_512
sbatch slurm/datagen.sbatch erosion_master/main_256
sbatch --cpus-per-task=32 --mem=128G --time=24:00:00 --output=logs/chunked-%j.out \
    --wrap "cd datagen && UV_OFFLINE=1 uv run --no-sync erosion-datagen chunked erosion_master/main_512 test --chunk 256"
```

**2. Train**, in this order. Each resumes if stopped; a finished model is skipped.

Local:

```sh
uv run erosion-sim train final_coarse_seed0
uv run erosion-sim train final_fine_seed0
uv run erosion-sim train erosion_final_seed0
```

Cluster:

```sh
sbatch slurm/train.sbatch final_coarse_seed0
sbatch slurm/train.sbatch final_fine_seed0
sbatch slurm/train.sbatch erosion_final_seed0
```

**3. Evaluate**, once training has finished. It refuses to replace existing results.

Local:

```sh
uv run erosion-sim evaluate world_all_runs final_seed0
uv run erosion-sim evaluate world_all_runs no_change
uv run erosion-sim evaluate world_all_runs analytical
uv run erosion-sim evaluate world_all_runs analytical_chunks_256
uv run erosion-sim evaluate world_all_runs analytical_blended_256
uv run erosion-sim evaluate world_all_runs erosion_final_seed0_chunks_256
uv run erosion-sim evaluate world_all_runs simulator_chunks_256
```

Cluster:

```sh
sbatch slurm/evaluate.sbatch world_all_runs final_seed0
sbatch slurm/evaluate.sbatch world_all_runs no_change
sbatch slurm/evaluate.sbatch world_all_runs analytical
sbatch slurm/evaluate.sbatch world_all_runs analytical_chunks_256
sbatch slurm/evaluate.sbatch world_all_runs analytical_blended_256
sbatch slurm/evaluate.sbatch world_all_runs erosion_final_seed0_chunks_256
sbatch slurm/evaluate.sbatch world_all_runs simulator_chunks_256
```

Then run the cells below.

How to read the tables and plots: [`docs/guides/world.md`](../../docs/guides/world.md).

In [ ]:
import json

from erosion_sim.erosion import plots
from erosion_sim.evaluation.results import load
from erosion_sim.paths import model_dir
from erosion_sim.training.train import plot_history

EVALUATION = "world_all_runs"  # configs/evaluations/world_all_runs.yaml
CASCADES = ["final_seed0"]  # configs/predictors/
BASELINES = ["no_change", "analytical", "analytical_chunks_256", "analytical_blended_256", "erosion_final_seed0_chunks_256", "simulator_chunks_256"]  # configs/predictors/
PREDICTORS = CASCADES + BASELINES
MODELS = ["final_coarse_seed0", "final_fine_seed0", "erosion_final_seed0"]  # every trained model these use, for the training plots

results = load(EVALUATION, *PREDICTORS)
{p: {k: m[k] for k in ("time", "code", "slurm_job")} for p, m in results.manifest["predictors"].items()}

## Training

In [ ]:
for name in MODELS:
    print(name, model_dir(name))
    plot_history(json.loads((model_dir(name) / "history.json").read_text()))

## Cascades, levels alone, and baselines, on the runs' interiors

In [ ]:
plots.table(results, plots.TABLE + ('chunk_edge_rmse', 'seconds_per_query'))

In [ ]:
if "dT" in results.scalars["by"]:
    plots.by_table(results, "dT", "skill", kinds=("cascade", "baseline"))

## Seams

How much neighboring windows disagree before blending, per level and phase, as a share of the change.

In [ ]:
for world, levels in results.scalars["seams"].items():
    for level, phases in levels.items():
        print(world, level, [f"{p['relative']:.1%}" for p in phases])

## Examples

In [ ]:
plots.plot_examples(results, kinds=("cascade", "baseline"))

## Conclusion

_Written after reading the results above._